In [1]:
import torch
from einops import rearrange, repeat, reduce, einsum, parse_shape
from einops.layers.torch import Rearrange
from torch import nn

In [2]:
x = torch.arange(24).reshape(2, 3, 4)
x

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [3]:
y = rearrange(x, "b t c -> b c t")
print(y.shape)
y

torch.Size([2, 4, 3])


tensor([[[ 0,  4,  8],
         [ 1,  5,  9],
         [ 2,  6, 10],
         [ 3,  7, 11]],

        [[12, 16, 20],
         [13, 17, 21],
         [14, 18, 22],
         [15, 19, 23]]])

In [4]:
y = rearrange(x, "b t c -> b (t c)")
print(y.shape)
y

torch.Size([2, 12])


tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]])

In [5]:
z = rearrange(y, "b (t c) -> b t c", c=4)
print(z.shape)
z

torch.Size([2, 3, 4])


tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [6]:
z = rearrange(y, "b (t c) -> b c t", c=4)
print(z.shape)
z

torch.Size([2, 4, 3])


tensor([[[ 0,  4,  8],
         [ 1,  5,  9],
         [ 2,  6, 10],
         [ 3,  7, 11]],

        [[12, 16, 20],
         [13, 17, 21],
         [14, 18, 22],
         [15, 19, 23]]])

In [7]:
y = rearrange(x, "b t c -> b (c t)")
print(y.shape)
y

torch.Size([2, 12])


tensor([[ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11],
        [12, 16, 20, 13, 17, 21, 14, 18, 22, 15, 19, 23]])

In [8]:
small = torch.arange(6).reshape(2, 3)
print(small)
print(rearrange(small, "a b -> (a b)"))
print(rearrange(small, "a b -> (b a)"))  # <- transposed first

tensor([[0, 1, 2],
        [3, 4, 5]])
tensor([0, 1, 2, 3, 4, 5])
tensor([0, 3, 1, 4, 2, 5])


In [9]:
x = torch.arange(24, dtype=torch.float).reshape(2, 3, 4)  # b=2, t=3, c=4
print(x)

tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]])


In [10]:
y = reduce(x, "b t c -> b c", 'mean')
print(y.shape)
print(y)
p = torch.mean(x, dim=1)
print(p.shape)
print(p)
print(torch.equal(y, p))

torch.Size([2, 4])
tensor([[ 4.,  5.,  6.,  7.],
        [16., 17., 18., 19.]])
torch.Size([2, 4])
tensor([[ 4.,  5.,  6.,  7.],
        [16., 17., 18., 19.]])
True


In [11]:
y = reduce(x, "b t c -> b", "max")
print(y.shape)
print(y)
p = torch.amax(x, dim=(1, 2))
print(p.shape)
print(p)
torch.equal(y, p)

torch.Size([2])
tensor([11., 23.])
torch.Size([2])
tensor([11., 23.])


True

In [12]:
y = reduce(x, "b t c -> b 1 c", "mean")
print(y.shape)
print(y)
p = torch.mean(x, dim=1, keepdim=True)
print(p.shape)
print(p)
torch.equal(y, p)

torch.Size([2, 1, 4])
tensor([[[ 4.,  5.,  6.,  7.]],

        [[16., 17., 18., 19.]]])
torch.Size([2, 1, 4])
tensor([[[ 4.,  5.,  6.,  7.]],

        [[16., 17., 18., 19.]]])


True

In [13]:
v = torch.tensor([1.0, 2.0, 3.0])
y = repeat(v, "c -> b c", b=2)
print(y.shape)
print(y)

torch.Size([2, 3])
tensor([[1., 2., 3.],
        [1., 2., 3.]])


In [14]:
print(repeat(v, "c -> c r", r=2))

tensor([[1., 1.],
        [2., 2.],
        [3., 3.]])


In [15]:
print(repeat(v, "c -> (c r)", r=2))

tensor([1., 1., 2., 2., 3., 3.])


In [16]:
print(repeat(v, "c -> (r c)", r=2))

tensor([1., 2., 3., 1., 2., 3.])


In [17]:
b, t, nh, hd = 2, 5, 4, 3
d = 10
x = torch.randn(b, t, nh, hd)
w = torch.randn(nh, hd, d)
torch.equal(
    einsum(x, w, "b t nh hd, nh hd d -> b t d"),
    rearrange(x, "b t nh hd -> b t (nh hd)") @ rearrange(w, "nh hd d -> (nh hd) d")
)

True

In [18]:
a = [torch.tensor(1), torch.tensor(2), torch.tensor(3), torch.tensor(4)]
x = rearrange(a, "(i j) -> i j", i=2)
y = rearrange(torch.stack(a), "(i j) -> i j", i=2)
print(x)
torch.equal(x, y)

tensor([[1, 2],
        [3, 4]])


True

In [19]:
x = torch.randn((2, 3, 4))
T = x.size(1)
t = parse_shape(x, 'b t c')['t']
print(parse_shape(x, 'b t c'))
assert T == t
t

{'b': 2, 't': 3, 'c': 4}


3

In [20]:
x = torch.randn((2, 3, 4))
print(parse_shape(x, '... c'))
print(parse_shape(x, 'b _ c'))

{'c': 4}
{'b': 2, 'c': 4}


In [21]:
n_embd = 8
qkv = torch.randn(2, 5, 3 * n_embd)  # b t (3*c), one Linear produced all three

q, k, v = rearrange(qkv, "b t (three c) -> three b t c", three=3)
q2, k2, v2 = qkv.split(n_embd, dim=-1)
print(torch.equal(q, q2), torch.equal(k, k2), torch.equal(v, v2))

True True True


In [22]:
import torch.nn.functional as F

b, t, nh, hd = 2, 5, 4, 3
q = torch.randn(b, nh, t, hd)
k = torch.randn(b, nh, t, hd)
v = torch.randn(b, nh, t, hd)

scores = q @ rearrange(k, "b nh t hd -> b nh hd t") * hd**-0.5

causal = torch.tril(torch.ones(t, t, dtype=torch.bool))
attn = scores.masked_fill(~causal, float("-inf")).softmax(dim=-1)

out = attn @ v

out = rearrange(out, "b nh t hd -> b t (nh hd)")

ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
assert out.shape == (b, t, nh * hd)

assert torch.allclose(out, rearrange(ref, "b nh t hd -> b t (nh hd)"), atol=1e-6)
print("ok")

ok


In [23]:
b, nh, t_past, hd = 2, 4, 6, 3

k_cache = torch.randn(b, nh, t_past, hd)
v_cache = torch.randn(b, nh, t_past, hd)

q_new = torch.randn(b, nh, 1, hd)
k_new = torch.randn(b, nh, 1, hd)
v_new = torch.randn(b, nh, 1, hd)

k_full = torch.cat([k_cache, k_new], dim=2)
v_full = torch.cat([v_cache, v_new], dim=2)
t = k_full.size(2)
scores = q_new @ rearrange(k_full, "b nh t hd -> b nh hd t") * hd**-0.5
tril = torch.tril(torch.ones(t, t, dtype=torch.bool))
causal = tril[t_past:t, :t]
scores = scores.masked_fill(causal == 0, float("-inf"))
attn = F.softmax(scores, dim=-1)
out = attn @ v_full

out = rearrange(out, "b nh t hd -> b t (nh hd)")

assert k_full.shape == (b, nh, t_past + 1, hd)

ref = F.scaled_dot_product_attention(q_new, k_full, v_full, is_causal=False)
assert out.shape == (b, 1, nh * hd)

assert torch.allclose(out, rearrange(ref, "b nh t hd -> b t (nh hd)"), atol=1e-6)
print("ok")

ok


In [24]:
x = torch.arange(24).reshape(2, 3, 4)
print(x)
print(reduce(x, "b t c -> b c", lambda tensor, axes: tensor.amax(axes)))

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
tensor([[ 8,  9, 10, 11],
        [20, 21, 22, 23]])


In [25]:
head_split = nn.Sequential(
    nn.Linear(6, 12),
    Rearrange("b t (nh hd) -> b nh t hd", nh=4),
)
print(head_split(torch.randn(2, 5, 6)).shape)  # (2, 4, 5, 3)
print(head_split)  # <- the reshape is right there in the repr

torch.Size([2, 4, 5, 3])
Sequential(
  (0): Linear(in_features=6, out_features=12, bias=True)
  (1): Rearrange('b t (nh hd) -> b nh t hd', nh=4)
)
